# Model experiments with MLflow

This notebook runs every recommender model with every data processor, logs the experiments to MLflow, and saves the processed datasets and model artifacts.

It is designed for repeatable experiment tracking and DVC-friendly dataset versioning.

In [1]:
from __future__ import annotations

import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import gc
from torch.utils.data import DataLoader, random_split
import yaml
import mlflow

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from recommender.data import DataProcessorContext, RecommenderDataset, load_events
from recommender.models import ModelFactory
from recommender.mlflow_toolkit import MLflowToolkit
from recommender.training import Trainer, hit_rate_at_k, ndcg_at_k, train_one_experiment
from recommender.training.early_stopping import EarlyStopping

In [2]:
CONFIG_PATH = ROOT / "configs" / "model.yaml"
MLFLOW_CONFIG_PATH = ROOT / "configs" / "mlflow.yaml"
PROCESSED_DIR = ROOT / "data" / "processed" / "mlflow_experiments"
ARTIFACT_DIR = ROOT / "models" / "mlflow_experiments"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    base_cfg = yaml.safe_load(f)["model"]

with open(MLFLOW_CONFIG_PATH) as f:
    mlflow_cfg = yaml.safe_load(f)["mlflow"]

processor_names = ["weighted", "binary", "implicit"]
model_types = ["ncf", "gmf", "matrix_factorization"]
total_runs = len(processor_names) * len(model_types)

processor_kwargs_map = {
    "weighted": {},
    "binary": {},
    "implicit": {},
}

mlflow_toolkit = MLflowToolkit(
    tracking_uri=mlflow_cfg.get("tracking_uri"),
    experiment_name=mlflow_cfg.get("experiment_name"),
)
mlflow_toolkit.setup()
print(f"MLflow tracking URI: {mlflow_cfg.get('tracking_uri')}")
print(f"Experiment: {mlflow_cfg.get('experiment_name')}")
print(f"Planned runs: {total_runs} ({len(model_types)} models x {len(processor_names)} processors)")

base_cfg


MLflow tracking URI: sqlite:///mlflow.db
Experiment: ecommerce_recommender-dev-2
Planned runs: 9 (3 models x 3 processors)


{'type': 'ncf',
 'seed': 42,
 'batch_size': 256,
 'learning_rate': 0.001,
 'epochs': 10,
 'show_progress': True,
 'num_negatives': 4,
 'min_interactions': 5,
 'raw_events_path': 'data/raw/events.csv',
 'artifact_dir': 'models',
 'processor': 'weighted',
 'processor_kwargs': {},
 'streaming': False,
 'early_stopping': {'enabled': True,
  'monitor': 'ndcg_at_k',
  'mode': 'max',
  'patience': 3,
  'min_delta': 0.0},
 'hyperparams': {'embedding_dim': 64,
  'hidden_layers': [128, 64, 32],
  'dropout': 0.2}}

In [3]:
def dvc_add(path: Path) -> None:
    try:
        relative_path = path.relative_to(ROOT)
    except ValueError:
        relative_path = path

    try:
        result = subprocess.run(
            ["poetry", "run", "dvc", "add", str(relative_path)],
            cwd=ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
        print(result.stdout.strip() or f"DVC tracked: {relative_path}")
    except FileNotFoundError:
        print(f"Poetry or DVC not available, skipped: dvc add {relative_path}")
    except subprocess.CalledProcessError as exc:
        print(f"DVC add failed for {relative_path}: {exc.stderr or exc.stdout}")


def prepare_processor_dataset(events: pd.DataFrame, processor_name: str) -> dict:
    print(f"Preparing processor dataset: {processor_name}")
    processor = DataProcessorContext(processor_name, **processor_kwargs_map[processor_name])
    interactions, user2idx, item2idx = processor.process(
        events,
        min_interactions=base_cfg.get("min_interactions", 1),
    )

    out_path = PROCESSED_DIR / f"{processor_name}_interactions.csv"
    interactions.to_csv(out_path, index=False)
    dvc_add(out_path)
    print(
        f"Saved {processor_name}: rows={len(interactions):,}, "
        f"users={len(user2idx):,}, items={len(item2idx):,}, path={out_path}"
    )

    return {
        "interactions": interactions,
        "user2idx": user2idx,
        "item2idx": item2idx,
        "path": out_path,
    }


def print_training_context(
    run_number: int,
    total_runs: int,
    model_type: str,
    processor_name: str,
    processor_data: dict,
) -> None:
    interactions = processor_data["interactions"]
    print("\n" + "=" * 72)
    print(f"Run {run_number}/{total_runs}")
    print(f"Model:     {model_type}")
    print(f"Dataset:   {processor_name} processor")
    print(f"Data file: {processor_data['path']}")
    print(
        f"Rows:      {len(interactions):,} | "
        f"Users: {len(processor_data['user2idx']):,} | "
        f"Items: {len(processor_data['item2idx']):,}"
    )
    print("=" * 72)

In [4]:
# DVC Integration Notes
# ======================
#
# This notebook uses DVC for data versioning:
# - Processed datasets are tracked with `dvc add` via the `dvc_add()` function
# - DVC uses `poetry run dvc` to ensure correct version is used
# - .dvc files are committed to Git to track data versions
# - Large data files are stored in DVC cache, not Git
#
# For experiment tracking with DVC:
# - Run: `poetry run dvc experiments run` (requires dvc.yaml configuration)
# - List: `poetry run dvc experiments list`
# - Compare: `poetry run dvc experiments show`
#
# Currently this notebook uses MLflow for experiment tracking, which is the primary
# experiment tracking system. DVC is used for data versioning only.
#
# train_one_experiment is now imported from recommender.training
# The function signature has been updated to accept:
# - config: ExperimentConfig dictionary
# - mlflow_toolkit: MLflowToolkit instance
# - artifact_dir: Path for saving artifacts

In [5]:
raw_events_path = ROOT / base_cfg.get("raw_events_path", "data/raw/events.csv")
print(f"Loading events from: {raw_events_path}")
events = load_events(str(raw_events_path))
print(f"Loaded events: {len(events):,}")

processor_cache = {
    processor_name: prepare_processor_dataset(events, processor_name)
    for processor_name in processor_names
}

# Build experiment config from base_cfg
experiment_config = {
    "batch_size": base_cfg["batch_size"],
    "epochs": base_cfg["epochs"],
    "learning_rate": base_cfg["learning_rate"],
    "num_negatives": base_cfg["num_negatives"],
    "show_progress": base_cfg.get("show_progress", True),
    "hyperparams": base_cfg.get("hyperparams", {}),
    "early_stopping_patience": base_cfg["early_stopping"]["patience"],
    "early_stopping_min_delta": base_cfg["early_stopping"]["min_delta"],
    "early_stopping_mode": base_cfg["early_stopping"]["mode"],
    "early_stopping_monitor": base_cfg["early_stopping"]["monitor"],
    "train_split_ratio": 0.8,
    "ranking_k": 10,
    "ranking_sample_limit": 10000,
    "ranking_positive_limit": 1000,
}

# Get the current experiment ID from your toolkit to restrict the search scope
current_experiment = mlflow.get_experiment_by_name(mlflow_cfg.get("experiment_name"))
experiment_id = current_experiment.experiment_id if current_experiment else "0"

results = []
run_number = 0
for model_type in model_types:
    for processor_name in processor_names:
        run_number += 1
        run_name = f"{model_type}-{processor_name}"
        print(f"=== Run {run_number}/{total_runs}: {run_name} ===")
        
        # ------------------------------------------------------------
        # Check if a successful run already exists
        # ------------------------------------------------------------
        filter_string = (
            f"tags.model_type = '{model_type}' AND "
            f"tags.processor = '{processor_name}' AND "
            f"attributes.status = 'FINISHED'"
        )
        
        existing_runs = mlflow.search_runs(
            experiment_ids=[experiment_id],
            filter_string=filter_string,
            max_results=1
        )
        
        if not existing_runs.empty:
            print(f"--> Skipping: Found an existing completed run for {run_name}.")
            # Optional: Extract the past metrics from existing_runs if you want them in results_df
            continue
        # ------------------------------------------------------------

        with mlflow_toolkit.start_run(
            run_name=run_name,
            tags={"model_type": model_type, "processor": processor_name},
        ):
            mlflow_toolkit.log_params({
                "model_type": model_type,
                "processor": processor_name,
                "seed": base_cfg["seed"],
                "batch_size": base_cfg["batch_size"],
                "learning_rate": base_cfg["learning_rate"],
                "epochs": base_cfg["epochs"],
                "num_negatives": base_cfg["num_negatives"],
                "min_interactions": base_cfg.get("min_interactions", 1),
            })
            result = train_one_experiment(
                processor_cache[processor_name],
                model_type,
                processor_name,
                config=experiment_config,
                mlflow_toolkit=mlflow_toolkit,
                artifact_dir=ARTIFACT_DIR,
                seed=base_cfg["seed"],
            )
            results.append(result)

if results:
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(["model_type", "processor"]).reset_index(drop=True)
else:
    print("\nAll planned runs were skipped because matching experiments already exist.")

Loading events from: /home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/events.csv
Loaded events: 2,756,101
Preparing processor dataset: weighted
DVC tracked: data/processed/mlflow_experiments/weighted_interactions.csv
Saved weighted: rows=897,028, users=81,318, items=67,625, path=/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/processed/mlflow_experiments/weighted_interactions.csv
Preparing processor dataset: binary
DVC tracked: data/processed/mlflow_experiments/binary_interactions.csv
Saved binary: rows=21,796, users=2,243, items=4,546, path=/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/processed/mlflow_experiments/binary_interactions.csv
Preparing processor dataset: implicit
DVC tracked: data/processed/mlflow_experiments/implicit_interactions.csv
Saved implicit: rows=897,028, users=81,318, items=67,625, path=/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/processed/mlflow_experiments/implicit_

/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema cont

=== Run 2/9: ncf-binary ===


=== Run 3/9: ncf-implicit ===


/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


...

=== Run 7/9: matrix_factorization-weighted ===


/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/src/recommender/training/experiment.py:169: UserWarning: Unexpected hyperparameters for model type 'matrix_facto

=== Run 8/9: matrix_factorization-binary ===


=== Run 9/9: matrix_factorization-implicit ===


/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/src/recommender/training/experiment.py:169: UserWarning: Unexpected hyperparameters for model type 'matrix_facto

## What gets saved

For each processor this notebook produces one dataframe saved under `data/processed/mlflow_experiments/` and tracked with `dvc add` when DVC is available.

For each `(model, processor)` run it logs MLflow params, per-epoch metrics, final ranking metrics, the input dataset reference, and the PyTorch checkpoint under `models/mlflow_experiments/`.

The progress output shows dataset preparation, run number, device, sample counts, per-epoch metrics, and final artifact paths so long-running experiments do not feel like a black box.
